In [128]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

In [129]:
from google.colab import files

uploaded = files.upload()

Saving data_testing.csv to data_testing (1).csv
Saving data_training.csv to data_training (2).csv


***Membaca Dataset***

In [130]:
train = pd.read_csv('data_training.csv')
test = pd.read_csv('data_testing.csv')

train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.3,0.740,0.08,1.7,0.094,10.0,45.0,0.99576,3.24,0.50,9.8,5,1366
1,8.1,0.575,0.22,2.1,0.077,12.0,65.0,0.99670,3.29,0.51,9.2,5,103
2,10.1,0.430,0.40,2.6,0.092,13.0,52.0,0.99834,3.22,0.64,10.0,7,942
3,12.9,0.500,0.55,2.8,0.072,7.0,24.0,1.00012,3.09,0.68,10.9,6,811
4,8.4,0.360,0.32,2.2,0.081,32.0,79.0,0.99640,3.30,0.72,11.0,6,918


In [131]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 857 entries, 0 to 856
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         857 non-null    float64
 1   volatile acidity      857 non-null    float64
 2   citric acid           857 non-null    float64
 3   residual sugar        857 non-null    float64
 4   chlorides             857 non-null    float64
 5   free sulfur dioxide   857 non-null    float64
 6   total sulfur dioxide  857 non-null    float64
 7   density               857 non-null    float64
 8   pH                    857 non-null    float64
 9   sulphates             857 non-null    float64
 10  alcohol               857 non-null    float64
 11  quality               857 non-null    int64  
 12  Id                    857 non-null    int64  
dtypes: float64(11), int64(2)
memory usage: 87.2 KB


In [132]:
train.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
count,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000
mean,8.261960,0.529393,0.267351,2.506184,0.086830,15.782964,45.978413,0.996692,3.313092,0.656709,10.430338,5.653442,813.749125
std,1.701992,0.179162,0.195144,1.293512,0.048721,10.300402,31.692113,0.001901,0.152079,0.167364,1.066971,0.821777,463.807063
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.390000,8.400000,3.000000,0.000000
25%,7.100000,0.395000,0.090000,1.900000,0.070000,7.000000,21.000000,0.995520,3.210000,0.550000,9.500000,5.000000,413.000000
50%,7.900000,0.520000,0.250000,2.200000,0.079000,14.000000,38.000000,0.996680,3.310000,0.620000,10.200000,6.000000,814.000000
75%,9.100000,0.640000,0.420000,2.600000,0.090000,21.000000,63.000000,0.997800,3.400000,0.730000,11.100000,6.000000,1214.000000
max,15.600000,1.580000,1.000000,15.500000,0.611000,68.000000,278.000000,1.003200,4.010000,2.000000,14.000000,8.000000,1597.000000


In [133]:
train['quality'].unique()

array([5, 7, 6, 4, 8, 3])

***Mengecek Missing Value***

In [134]:
train.isnull().sum()

,0
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


Berdasarkan pengecekan missing value menggunakan isnull().sum(), seluruh variabel memiliki nilai 0. Hal ini menunjukkan bahwa dataset tidak memiliki data yang hilang (missing values), sehingga tidak diperlukan proses imputasi maupun penghapusan data.

In [135]:
X = train.drop('quality', axis=1)
y = train['quality']

***Split Data Training dan Validation***

In [136]:
train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

[     fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
 177            9.0             0.785         0.24            1.70      0.078   
 505            7.2             0.725         0.05            4.65      0.086   
 17             7.7             0.490         0.26            1.90      0.062   
 738            8.0             0.380         0.44            1.90      0.098   
 236            7.4             0.580         0.00            2.00      0.064   
 ..             ...               ...          ...             ...        ...   
 533           10.2             0.290         0.65            2.40      0.075   
 374            8.2             0.260         0.34            2.50      0.073   
 174            8.2             0.730         0.21            1.70      0.074   
 18             9.1             0.765         0.04            1.60      0.078   
 468           10.8             0.320         0.44            1.60      0.063   
 
      free sulfur dioxide 

***Feature Scaling***

Berdasarkan hasil eksplorasi data, beberapa fitur memiliki rentang nilai yang berbeda cukup jauh. Sebagai contoh, variabel total sulfur dioxide memiliki nilai yang jauh lebih besar dibandingkan variabel pH atau chlorides. Oleh karena itu dilakukan pengecekan kebutuhan feature scaling.

Namun, model yang digunakan adalah Random Forest Classifier yang berbasis decision tree sehingga tidak terlalu sensitif terhadap perbedaan skala data. Oleh sebab itu, model tetap dapat bekerja dengan baik tanpa normalisasi tambahan.

In [137]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

***Membuat Model Random Forest***

In [138]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

Pada penelitian ini digunakan model Random Forest dengan parameter `n_estimators=300`, `random_state=42`, dan `class_weight='balanced'`. Parameter `n_estimators=300` menunjukkan bahwa model membangun 300 decision tree untuk meningkatkan stabilitas dan performa prediksi. Parameter `random_state=42` digunakan agar hasil pelatihan model tetap konsisten setiap kali dijalankan, sedangkan `class_weight='balanced'` digunakan untuk mengatasi ketidakseimbangan data karena jumlah data pada setiap kelas kualitas wine tidak sama. Proses `model.fit(X_train_scaled, y_train)` digunakan untuk melatih model menggunakan data training yang telah melalui proses scaling sehingga model dapat mempelajari hubungan antara karakteristik kimiawi wine dengan nilai kualitas (`quality`) dan selanjutnya digunakan untuk melakukan prediksi pada data baru.


***Prediksi validation***

Proses y_pred = model.predict(X_valid_scaled) digunakan untuk menghasilkan prediksi nilai quality pada data validasi. Hasil prediksi tersebut kemudian dibandingkan dengan data aktual (y_valid) untuk mengevaluasi performa model menggunakan accuracy, confusion matrix, dan classification report.

In [139]:
y_pred = model.predict(X_valid_scaled)

***Accuracy***

In [140]:
accuracy = accuracy_score(y_valid, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.6511627906976745


Berdasarkan hasil evaluasi model Random Forest Classifier, diperoleh nilai accuracy sebesar 0,65 atau sekitar 65%. Hal ini menunjukkan bahwa model mampu memprediksi kualitas wine dengan benar sebanyak 65% dari total data validasi.

***Confusion Matrix***

In [141]:
cm = confusion_matrix(y_valid, y_pred)

print(cm)

[[ 0  1  2  0  0]
 [ 0 49 18  0  0]
 [ 0 22 54  2  0]
 [ 0  1 11  9  0]
 [ 0  0  0  3  0]]


Confusion matrix menunjukkan perbandingan antara hasil prediksi model dengan data aktual pada setiap kelas kualitas wine. Baris pada confusion matrix menunjukkan kelas sebenarnya, sedangkan kolom menunjukkan hasil prediksi model. Berdasarkan hasil yang diperoleh, model mampu memprediksi dengan cukup baik pada kelas kualitas 5 dan 6 karena jumlah prediksi benar pada kedua kelas tersebut paling tinggi dibandingkan kelas lainnya. Namun, model masih mengalami kesalahan prediksi terutama antara kelas 5, 6, dan 7 karena karakteristik data antar kelas cukup mirip. Selain itu, model belum mampu memprediksi kelas 4 dan 8 dengan baik karena jumlah data pada kedua kelas tersebut sangat sedikit sehingga pola data sulit dipelajari oleh model. Secara keseluruhan, confusion matrix menunjukkan bahwa model Random Forest sudah cukup baik dalam mengenali kelas mayoritas, tetapi masih kurang optimal pada kelas minoritas akibat ketidakseimbangan data (imbalanced dataset).


***Classification Report***

In [142]:
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

           4       0.00      0.00      0.00         3
           5       0.67      0.73      0.70        67
           6       0.64      0.69      0.66        78
           7       0.64      0.43      0.51        21
           8       0.00      0.00      0.00         3

    accuracy                           0.65       172
   macro avg       0.39      0.37      0.38       172
weighted avg       0.63      0.65      0.64       172



Accuracy = 0.65

Model memiliki tingkat ketepatan prediksi sebesar 65%.

*Precision*

Precision menunjukkan seberapa tepat prediksi model pada suatu kelas.

Contoh:

precision kelas 5 = 0.67
artinya dari seluruh prediksi kualitas 5, sekitar 67% benar.

*Recall*

Recall menunjukkan kemampuan model menemukan data dari kelas tertentu.

Contoh:

recall kelas 5 = 0.73
artinya 73% data kualitas 5 berhasil dikenali model.

*F1-Score*

F1-score adalah gabungan precision dan recall.

Nilai tertinggi terdapat pada:

kelas 5 → 0.70
kelas 6 → 0.66

Artinya model paling baik memprediksi kualitas 5 dan 6.

*Macro Average dan Weighted Average*

Macro Average = 0.38

Rata-rata performa seluruh kelas masih rendah karena kelas minoritas (4 dan 8) tidak terprediksi dengan baik.

Weighted Average = 0.64

Menunjukkan performa keseluruhan model cukup baik karena dipengaruhi oleh kelas mayoritas yaitu kualitas 5 dan 6.

***Prediksi Data Testing***

In [143]:
X_test = test.copy()

X_test_scaled = scaler.transform(X_test)

In [144]:
model.fit(X_train_scaled, y_train)
test_pred = model.predict(X_test_scaled)

***Membuat File Submission***

In [123]:
submission = pd.DataFrame({
    'Id': test['Id'],
    'Quality': test_pred
})

In [124]:
submission.to_csv(
    'hasilprediksi.csv',
    index=False,
    sep=';'
)
print("File berhasil disimpan")

File berhasil disimpan


In [125]:
files.download('hasilprediksi_018.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>